# Machine Learning for Chemistry

A key challenge for machine learning is chemistry is to convert molecular structures into formats that can be understood by computers. Chemists typically convey information about molecular structure through skeletal drawings that include a lot of implicit information, such as:
* Atomic composition
* Atomic connectivity
* Stereochemistry 
* Bond orders
* Formal charge

This information is readily understood by humans but can be tricky to encode in computer-friendly representations. 

# Molecular Representations

#### SMILES
SMILES (simplified molecular input entry line system) strings are text-based representations of molecular structure that are commonly used to represent molecules in various databases. They provide a complete picture of the connectivity of a molecule and can be used to encode *some* spatial orientation of atoms. 

The image below shows how a SMILES string is related to skeletal molecular structure. One structure can be represented by many different SMILES strings, meaning  they are not *invariant* representations. This is an important consideration for machine learning, as two SMILES strings for the same molecule could produce different outputs if used as the input for a predictive model. Despite this, some generative models can be trained to produce SMILES strings for molecular discovery. 

![smiles](https://upload.wikimedia.org/wikipedia/commons/0/00/SMILES.png)

Use the visualiser code below to inspect some of the SMILES strings for common drugs:
* Morphine: CN1CC[C@@]23[C@H]4OC5=C2C(C[C@@H]1[C@@H]3C=C[C@@H]4O)=CC=C5O
* Ibuprofen:  CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O
* Penicillin: CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C



In [22]:
from ipywidgets import interact

from rdkit import Chem
from rdkit.Chem import AllChem

import py3Dmol

smiles = "CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C" # place smiles here

mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

AllChem.EmbedMolecule(mol)
AllChem.MMFFOptimizeMolecule(mol)
mol_block = Chem.MolToMolBlock(mol)

port = py3Dmol.view()
port.addModel(mol_block)
port.setStyle({'model': -1}, {"stick": {'radius': 0.15}, "sphere": {"colorscheme": "Jmol", 'radius': .4}})
port.zoomTo()
port.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

#### Molecular Fingerprints

An alternative molecular representation is a *molecular fingerprint*, a binary array that describes the substructures present within a molecule. 

There are different ways of generating molecular fingerprints. the simplest is to use a pre-defined list of substructures ([MACCS implementation](https://github.com/rdkit/rdkit/blob/master/rdkit/Chem/MACCSkeys.py)), but this can miss important atoms. An alternative is to construct Extended Connectivity FingerPrints (ECFPs) which account for the local environment of each atom and use a hashing function to encode their positions within the fingerprint array. 

Unlike with SMILEs strings, a molecular fingerprint cannot be used to derive it molecular structure.  Another disadvantage of ECFPs is the tendency for certain molecular environments to be hashed down to the same bit in a binary fingerprint array. These are called bit collisions. 

----

# Solubility Task

For a molecule to act effectively as a drug, it must be soluble under physiological conditions. Therefore, the solubility of a compound must be known before it is assessed as a drug candidate. We can use machine learning for such tasks to try and predict the solubility of a drug based on its molecular structure. 

For this task, we will train a machine learning model to predict a molecules solubility from its molecular structure using ECFPs. The code below loads in the data from the [Therapeutics Data Commons](https://tdc.readthedocs.io/en/main/) ADME library. 

You will need to use the following functions from RDKit, a python library specifically designed for cheminformatics:

* `mol = Chem.MolFromSmiles(smiles) # loads in a SMILES string and converts it to and RDKit molecule object`
* `fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024) # converts RDKit molecule object into morgan firngerprint`

You will need to: 
* Train an ML model to predict molecular solubility from its ECFP with the highest possible accuracy


In [34]:
from tdc.single_pred.adme import ADME
import pandas as pd

solubility_data = ADME('Solubility_AqSolDB')
solubility_split = solubility_data.get_split() # use predefined splits 

train_df = pd.concat([solubility_split["train"], solubility_split["valid"] ])
test_df = solubility_split["test"]

# convert dataframes to np arrays 
train_smiles = train_df["Drug"].to_numpy()
train_solubility = train_df["Y"].to_numpy()

test_smiles = test_df["Drug"].to_numpy()
test_solubility = test_df["Y"].to_numpy()

... # your code here

Found local copy...
Loading...
Done!
